# Notebook para estudos de vector search   

### Vector Similatiry   
**Similatiry Metrics**   
* Cosine similarity    

#### Vecotr Search Strategies   
* k-nearest neighbors (KNN)    
</br>
* Approximated Nearest Neighbors (ANN)
  * Trade accuracy for speed gains
  * Examples of index algorihms:
    * Tree-based: ANNOY by Spotify
    * Proximity graphs: HNSW
    * Clustering: FAISS by Facebook
    * Hashing: LSH
    * Vector compression: SCaNN by Google, Product Quantization (PQ)

#### Comparação de tempo de busca   

Comparando cosine similarity com e sem LSH.

In [0]:
!pip install datasketch

In [0]:
import numpy as np
import time
from datasketch import MinHashLSH, MinHash

In [0]:
# Cria uma função para gerar vetores semelhantes
def create_similar_vectors(base_vector, num_vectors=10, similarity_level=0.9):
    similar_vectors = []
    num_dimensions = len(base_vector)
    for _ in range(num_vectors):
        # Cria um vetor com a maioria dos elementos iguais ao vetor base
        similar_vector = np.copy(base_vector)
        # Inverte uma pequena porcentagem dos elementos para simular diferença
        num_changes = int(num_dimensions * (1 - similarity_level))
        change_indices = np.random.choice(num_dimensions, num_changes, replace=False)
        similar_vector[change_indices] = 1 - similar_vector[change_indices]
        similar_vectors.append(similar_vector)
    return similar_vectors

# Cria uma base de dados grande de vetores
data_size = 10000
vector_dim = 10000
vectors = []
for _ in range(data_size):
    # Vetores aleatórios (binários para simplificar)
    vectors.append(np.random.randint(2, size=vector_dim))

# Vamos adicionar um grupo de vetores muito semelhantes
base_vector_to_find = np.random.randint(2, size=vector_dim)
similar_group = create_similar_vectors(base_vector_to_find, num_vectors=20)
vectors.extend(similar_group)

# Nosso vetor de busca
query_vector = similar_group[0]

In [0]:
def brute_force_search(query_vector, data, threshold=0.9):
    similar_items = []
    # Normalização dos vetores para o cálculo do cosseno
    query_norm = np.linalg.norm(query_vector)
    
    for item in data:
        item_norm = np.linalg.norm(item)
        
        if query_norm > 0 and item_norm > 0:
            similarity = np.dot(query_vector, item) / (query_norm * item_norm)
            if similarity >= threshold:
                similar_items.append(item)
    return similar_items

print("Iniciando busca por força bruta...")
start_time = time.time()
brute_force_results = brute_force_search(query_vector, vectors)
end_time = time.time()
print(f"Tempo de busca por força bruta: {end_time - start_time:.4f} segundos")
print(f"Itens encontrados (força bruta): {len(brute_force_results)}")

In [0]:
# Create MinHash signatures for each vector
num_permutations = 128

def get_minhash_from_vector(vector):
    m = MinHash(num_perm=num_permutations)
    for i, val in enumerate(vector):
        if val == 1:
            m.update(str(i).encode('utf8'))
    return m

# Create and populate the LSH object
# The constructor only needs the `threshold` and `num_permutations` of the MinHash objects.
# The LSH will automatically determine the number of bands and rows.
lsh = MinHashLSH(threshold=0.8)

start_time_lsh_index = time.time()
minhashes = {}
for i, vector in enumerate(vectors):
    m = get_minhash_from_vector(vector)
    minhashes[f"vector_{i}"] = m
    lsh.insert(f"vector_{i}", m)
end_time_lsh_index = time.time()
print(f"\nTempo de indexação LSH: {end_time_lsh_index - start_time_lsh_index:.4f} segundos")

# Cria a assinatura do vetor de busca
query_minhash = get_minhash_from_vector(query_vector)

print("Iniciando busca com LSH...")
start_time_lsh = time.time()
lsh_results_keys = lsh.query(query_minhash)
end_time_lsh = time.time()
print(f"Tempo de busca com LSH: {end_time_lsh - start_time_lsh:.4f} segundos")
print(f"Itens encontrados (LSH): {len(lsh_results_keys)}")

# Let's also verify that some of the results are correct by calculating the similarity
print("\nVerificando resultados LSH:")
# We'll calculate the true similarity of the first 5 results found by LSH
for key in lsh_results_keys[:5]:
    true_similarity = query_minhash.jaccard(minhashes[key])
    print(f"  Similaridade com {key}: {true_similarity:.4f}")